# Text Preprocessing

## Instalasi Library (Jika Diperlukan)

In [2]:
# %pip install nltk pandas

## 1. Persiapan Data (Data Collection)
- **Tujuan:** Mendapatkan dataset yang memadai untuk pelatihan model.
- Memuat data menggunakan `pandas` dan melakukan pengecekan awal.

In [3]:
import pandas as pd

# Load dataset
df = pd.read_csv('../Dataset/fake_news_dataset.csv')
df.head()

,title,text,date,source,author,category,label
0,Foreign Democrat final.,more tax development both store agreement lawy...,2023-03-10,NY Times,Paula George,Politics,real
1,To offer down resource great point.,probably guess western behind likely next inve...,2022-05-25,Fox News,Joseph Hill,Politics,fake
2,Himself church myself carry.,them identify forward present success risk sev...,2022-09-01,CNN,Julia Robinson,Business,fake
3,You unit its should.,phone which item yard Republican safe where po...,2023-02-07,Reuters,Mr. David Foster DDS,Science,fake
4,Billion believe employee summer how.,wonder myself fact difficult course forget exa...,2023-04-03,CNN,Austin Walker,Technology,fake


In [4]:
# Pengecekan missing values
print("Missing Values:\n", df.isnull().sum())

# Pengecekan distribusi kelas (Fake/Real)
print("\nDistribusi Label:\n", df['label'].value_counts())

Missing Values:
 title          0
text           0
date           0
source      1000
author      1000
category       0
label          0
dtype: int64

Distribusi Label:
 label
fake    10056
real     9944
Name: count, dtype: int64


## 2. Text Preprocessing
Membersihkan teks berita agar dapat diproses oleh algoritma Machine Learning dengan optimal.

### 2.1 Case Folding
Mengubah seluruh teks menjadi huruf kecil (lowercase) dan menghapus karakter khusus, angka, serta tautan (URL).

In [5]:
import re

def case_folding(text):
    # Mengubah ke lowercase
    text = str(text).lower()
    # Menghapus URL
    text = re.sub(r'http\S+', '', text)
    # Menghapus karakter selain huruf dan spasi (menghapus tanda baca dan angka)
    text = re.sub(r'[^a-z\s]', '', text)
    # Menghapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Terapkan pada kolom text dan title
df['text_clean'] = df['text'].apply(case_folding)
df['title_clean'] = df['title'].apply(case_folding)

df[['text', 'text_clean']].head()

,text,text_clean
0,more tax development both store agreement lawy...,more tax development both store agreement lawy...
1,probably guess western behind likely next inve...,probably guess western behind likely next inve...
2,them identify forward present success risk sev...,them identify forward present success risk sev...
3,phone which item yard Republican safe where po...,phone which item yard republican safe where po...
4,wonder myself fact difficult course forget exa...,wonder myself fact difficult course forget exa...


### 2.2 Tokenization
Memecah teks utuh menjadi sekumpulan token (kata).

In [6]:
df['tokens'] = df['text_clean'].apply(lambda x: x.split())
df[['text_clean', 'tokens']].head()

,text_clean,tokens
0,more tax development both store agreement lawy...,"[more, tax, development, both, store, agreemen..."
1,probably guess western behind likely next inve...,"[probably, guess, western, behind, likely, nex..."
2,them identify forward present success risk sev...,"[them, identify, forward, present, success, ri..."
3,phone which item yard republican safe where po...,"[phone, which, item, yard, republican, safe, w..."
4,wonder myself fact difficult course forget exa...,"[wonder, myself, fact, difficult, course, forg..."


### 2.3 Stopword Removal
Menghapus kata-kata umum yang tidak memberikan konteks signifikan dalam bahasa Inggris.

In [7]:
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

df['tokens'] = df['tokens'].apply(lambda tokens: [word for word in tokens if word not in stop_words])
df['tokens'].head()

0    [tax, development, store, agreement, lawyer, h...
1    [probably, guess, western, behind, likely, nex...
2    [identify, forward, present, success, risk, se...
3    [phone, item, yard, republican, safe, police, ...
4    [wonder, fact, difficult, course, forget, exac...
Name: tokens, dtype: object

### 2.4 Stemming / Lemmatization
Mengembalikan kata ke bentuk dasarnya. Pada kasus ini kita menggunakan `PorterStemmer` dari NLTK.

In [8]:
from nltk.stem import PorterStemmer

ps = PorterStemmer()
df['tokens'] = df['tokens'].apply(lambda tokens: [ps.stem(word) for word in tokens])
df['tokens'].head()

0    [tax, develop, store, agreement, lawyer, hear,...
1    [probabl, guess, western, behind, like, next, ...
2    [identifi, forward, present, success, risk, se...
3    [phone, item, yard, republican, safe, polic, i...
4    [wonder, fact, difficult, cours, forget, exact...
Name: tokens, dtype: object

### 2.5 Menggabungkan Kembali Teks (Rejoining Tokens)
Menggabungkan kembali token yang sudah bersih dan di-stem menjadi string teks utuh (`clean_text`) untuk digunakan pada tahap Ekstraksi Fitur (Feature Extraction).

In [9]:
df['clean_text'] = df['tokens'].apply(lambda x: ' '.join(x))
df[['text', 'clean_text']].head()

,text,clean_text
0,more tax development both store agreement lawy...,tax develop store agreement lawyer hear outsid...
1,probably guess western behind likely next inve...,probabl guess western behind like next invest ...
2,them identify forward present success risk sev...,identifi forward present success risk sever fr...
3,phone which item yard Republican safe where po...,phone item yard republican safe polic identifi...
4,wonder myself fact difficult course forget exa...,wonder fact difficult cours forget exactli pat...


### 2.6 Isi value kosong di kolom source dan author dengan unknown


In [10]:
# Isi value kosong di kolom source dan author dengan unknown
df['author'] = df['author'].fillna('Unknown')
df['source'] = df['source'].fillna('Unknown')

print(f"Jumlah data setelah dibersihkan: {len(df)}")

Jumlah data setelah dibersihkan: 20000


## 3. Eksplorasi Hasil Preprocessing
Melakukan observasi sederhana pada hasil teks yang sudah dibersihkan.

In [11]:
# Perbandingan jumlah kata (Word Count) sebelum dan sesudah preprocessing
df['word_count_original'] = df['text'].apply(lambda x: len(str(x).split()))
df['word_count_clean'] = df['clean_text'].apply(lambda x: len(x.split()))

df[['word_count_original', 'word_count_clean']].describe()

,word_count_original,word_count_clean
count,20000.000000,20000.000000
mean,250.183300,223.949650
std,29.079644,26.486955
min,200.000000,167.000000
25%,225.000000,201.000000
50%,250.000000,224.000000
75%,275.000000,246.000000
max,300.000000,283.000000


In [12]:
# Cek 20 kata yang paling sering muncul pada dataset setelah dibersihkan
from collections import Counter

all_words = ' '.join(df['clean_text']).split()
most_common_words = Counter(all_words).most_common(20)

print("20 Kata Paling Sering Muncul:")
for word, count in most_common_words:
    print(f"{word}: {count}")

20 Kata Paling Sering Muncul:
manag: 15589
differ: 10501
natur: 10453
even: 10451
success: 10403
product: 10398
person: 10367
like: 10364
gener: 10337
build: 10334
cultur: 10321
develop: 10309
polit: 10306
discuss: 10304
perform: 10298
democrat: 10279
meet: 10265
final: 10264
offic: 10263
posit: 10260


In [13]:
# Menyimpan dataset yang sudah bersih ke file baru (opsional)
df.to_csv('../Dataset/fake_news_dataset_cleaned.csv', index=False)
print("Dataset bersih berhasil disimpan ke 'Dataset/fake_news_dataset_cleaned.csv'")

Dataset bersih berhasil disimpan ke 'Dataset/fake_news_dataset_cleaned.csv'


In [14]:
df.head()

,title,text,date,source,author,category,label,text_clean,title_clean,tokens,clean_text,word_count_original,word_count_clean
0,Foreign Democrat final.,more tax development both store agreement lawy...,2023-03-10,NY Times,Paula George,Politics,real,more tax development both store agreement lawy...,foreign democrat final,"[tax, develop, store, agreement, lawyer, hear,...",tax develop store agreement lawyer hear outsid...,216,192
1,To offer down resource great point.,probably guess western behind likely next inve...,2022-05-25,Fox News,Joseph Hill,Politics,fake,probably guess western behind likely next inve...,to offer down resource great point,"[probabl, guess, western, behind, like, next, ...",probabl guess western behind like next invest ...,238,219
2,Himself church myself carry.,them identify forward present success risk sev...,2022-09-01,CNN,Julia Robinson,Business,fake,them identify forward present success risk sev...,himself church myself carry,"[identifi, forward, present, success, risk, se...",identifi forward present success risk sever fr...,222,200
3,You unit its should.,phone which item yard Republican safe where po...,2023-02-07,Reuters,Mr. David Foster DDS,Science,fake,phone which item yard republican safe where po...,you unit its should,"[phone, item, yard, republican, safe, polic, i...",phone item yard republican safe polic identifi...,247,228
4,Billion believe employee summer how.,wonder myself fact difficult course forget exa...,2023-04-03,CNN,Austin Walker,Technology,fake,wonder myself fact difficult course forget exa...,billion believe employee summer how,"[wonder, fact, difficult, cours, forget, exact...",wonder fact difficult cours forget exactli pat...,215,195


In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   title                20000 non-null  str   
 1   text                 20000 non-null  str   
 2   date                 20000 non-null  str   
 3   source               20000 non-null  str   
 4   author               20000 non-null  str   
 5   category             20000 non-null  str   
 6   label                20000 non-null  str   
 7   text_clean           20000 non-null  str   
 8   title_clean          20000 non-null  str   
 9   tokens               20000 non-null  object
 10  clean_text           20000 non-null  str   
 11  word_count_original  20000 non-null  int64 
 12  word_count_clean     20000 non-null  int64 
dtypes: int64(2), object(1), str(10)
memory usage: 2.0+ MB
